# 01 — Exploratory Data Analysis

**This notebook contains no analysis logic of its own.** Every function it calls
lives in `src/battery_rul/` and is the same code the pipeline runs. That is
deliberate: a notebook that reimplements the pipeline is a second, untested
codebase that will silently drift.

Run `python scripts/download_data.py` first if `data/raw/nasa/mat/` is empty —
otherwise the loader falls back to the synthetic generator and says so loudly.

In [ ]:
%load_ext autoreload
%autoreload 2

from battery_rul.config import load_config
from battery_rul.data import load_cycles, battery_summary_table, schema_frame
from battery_rul.visualization import apply_style, generate_eda_figures

cfg = load_config('../configs/default.yaml')
apply_style(cfg.viz)
cfg.experiment_name, cfg.data.source, cfg.eol_capacity_ah

## The canonical schema

Every dataset — NASA today, CALCE tomorrow — is normalised into one row per
discharge cycle with these columns. Modelling code is written against this
contract and nothing else.

In [ ]:
schema_frame()

## Load and validate

`load_cycles` runs the full ingestion path: parse, trim leading rig artifacts,
derive health, validate, apply the beginning-of-life cohort gates, cache.

In [ ]:
dataset = load_cycles(cfg)
print(dataset.metadata.to_dict()['notes'])
print(dataset.validation.summary())
dataset.frame.shape

## Which cells survived, and why the others did not

The config asks for *every* cell on disk. The cohort is selected mechanically
by the gates, not hand-picked — see `docs/dataset_card.md` for the accounting.

In [ ]:
battery_summary_table(dataset, cfg)

## Validation findings

In [ ]:
import pandas as pd
pd.DataFrame([i.to_dict() for i in dataset.validation.issues])[['check','severity','message']]

## Capacity degradation

The faint lines are raw measurements; the bold lines are the **trailing** median.
The non-monotonicity is not noise — it is real capacity recovery after rest
periods, which is exactly why the end-of-life rule requires a *persistent*
threshold crossing rather than a first crossing.

In [ ]:
from battery_rul.visualization.style import battery_palette, figure

df = dataset.frame
cells = sorted(df.battery_id.unique())
colours = battery_palette(cells)

with figure(figsize=(11, 6), cfg=cfg.viz) as (fig, ax):
    for c in cells:
        g = df[df.battery_id == c]
        ax.plot(g.cycle_index, g.capacity_ah, alpha=.25, color=colours[c], lw=.9)
        ax.plot(g.cycle_index, g.capacity_smooth_ah, color=colours[c], lw=2, label=c)
    ax.axhline(cfg.eol_capacity_ah, ls='--', color='#B00020')
    ax.set_xlabel('Discharge cycle'); ax.set_ylabel('Capacity (Ah)')
    ax.set_title('Capacity fade; dashed line is end of life')
    ax.legend(ncol=4)

## Correlation between raw signals

Note how strongly the charge-timing and resistance signals track capacity.
That collinearity is real physics, and it is why feature importances later on
are read at the level of *signal families* rather than individual columns.

In [ ]:
import numpy as np
cols = ['capacity_ah','soh','internal_resistance_ohm','cc_ct_ratio',
        'voltage_min_v','temperature_max_c','discharge_duration_s','energy_throughput_wh']
df[[c for c in cols if c in df.columns]].corr().round(3)

## The full publication-quality figure set

Written to `figures/eda/`. These are the same figures the pipeline generates.

In [ ]:
paths = generate_eda_figures(dataset.frame, cfg, outdir=cfg.paths.figures_dir / 'eda')
[p.name for p in paths]

---
**Next:** `02_feature_engineering.ipynb` — the RUL target and the causality proof.